# If on Colab, run the following:

In [ ]:
! git clone https://github.com/ketchicken/cse144-spring-2026-final-project.git

In [ ]:
import sys
import importlib
sys.path.insert(0, '/content/cse144-spring-2026-final-project')
cse144spring2026finalproject = importlib.import_module("cse144-spring-2026-final-project")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# If not on Colab, start running cells from here. Make sure to revise file paths as necessary

In [ ]:
# Path to best weights
ckpt_path = "/content/drive/MyDrive/cse144_final_project_checkpoints/final_05_best_01_resultsfold0"

# Name of Output File
output_file = "submission.csv"
corrected_output_file = "corrected_submission.csv"

# Path To Test Data
data_dir = ''

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2 as tfv2
from torchvision.datasets import ImageFolder
from test import test_model, remap_values
import csv
from load_data import TestSet
from setup_env import set_seed
from model import ENetV2
set_seed(42)

DEVICE =  torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0

# RUN FOLLOWING CELL TO GENERATE SUBMISSION AND CORRECTED SUBMISSION

In [ ]:
# Testing, save results as .csv file with imgID | class

# Load the model
model = ENetV2().to(DEVICE)

state_dict = torch.load(ckpt_path, map_location='cpu')['model_state_dict']
model.load_state_dict(state_dict)
model.eval()

test_transforms = tfv2.Compose([
    # Normalization
    tfv2.Resize((480,480), interpolation=tfv2.InterpolationMode.BICUBIC),
    tfv2.CenterCrop((480,480)),
    tfv2.ToImage(),
    tfv2.ToDtype(torch.float32, scale=True),
    tfv2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = ImageFolder(root_dir=data_dir+'train') # just to grab the original order of class labels
test_set = TestSet(root_dir=data_dir+'test', transform=test_transforms)
test_loader = DataLoader(test_set, batch_size=1, num_workers=NUM_WORKERS, shuffle=False)

test_model(test_loader, model, DEVICE, outfile=output_file)

idx_to_class = {str(v):k for k, v in train_set.class_to_idx.items()} # Remapping index to class labels
remap_values(output_file, "corrected_submission.csv", idx_to_class)